# Lab 25-07 · Interposers, Hybrid-Bond Yield and Chiplet Cost

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import chiplets as c

## Part 1 · Interposers per wafer

*Predict* how many 50 mm × 65 mm interposers fit on a 300 mm wafer.

In [2]:
n = c.sites_on_wafer(50, 65)
A = 50 * 65
formula = math.pi * 300**2 / (4 * A) - math.pi * 300 / math.sqrt(2 * A)
print(f"exact count {n}; 25-04 formula {formula:.1f}")
sides = np.arange(25, 81, 5)
plt.figure()
plt.plot(sides**2, [c.sites_on_wafer(s, s, steps=20) for s in sides], "o-", label="exact (square)")
plt.plot(sides**2, [math.pi*300**2/(4*s*s) - math.pi*300/math.sqrt(2*s*s) for s in sides], "--", label="25-04 formula")
plt.xlabel("interposer area (mm2)"); plt.ylabel("per 300 mm wafer"); plt.legend(); plt.grid(True)
plt.show()

exact count 14; 25-04 formula 10.1


## Part 2 · Bond yield

*Predict* the die bond yield at 6 µm pitch with σ = 0.6 µm per axis.

In [3]:
y = c.die_bond_yield(6, 0.6)
ys = c.simulate_bond_yield(6, 0.6)
print(f"formula {100*y:.2f} %, Monte Carlo {100*ys:.2f} %")
r = np.linspace(0.02, 0.2, 60)
plt.figure()
plt.plot(r, [100 * c.die_bond_yield(1, x) for x in r])
plt.xlabel("placement sigma / pitch"); plt.ylabel("die bond yield (%)"); plt.grid(True)
plt.show()

formula 97.53 %, Monte Carlo 97.53 %


## Part 3 · Clustered defects

*Predict* how clustering (α = 2) moves the break-even defect density of Example 25-07.9.

In [4]:
be_p = c.breakeven_d0(800, 4, 0.10, 0.30, 3, 50, 110, 0.99)
be_c = c.breakeven_d0(800, 4, 0.10, 0.30, 3, 50, 110, 0.99, alpha=2)
print(f"break-even D0: Poisson {be_p:.3f}, clustered {be_c:.3f} per cm2")

break-even D0: Poisson 0.058, clustered 0.067 per cm2


## Part 4 · Break-even area

*Predict* the logic area at which four chiplets match one die at D0 = 0.1 per cm².

In [5]:
from scipy import optimize
def gap(a):
    return (c.good_package_cost([a], 0.1, 0.30, 3, 50, 0.99)
            - c.good_package_cost([a / 4 * 1.1] * 4, 0.1, 0.30, 3, 110, 0.99))
print(f"break-even area {optimize.brentq(gap, 100, 1500):.0f} mm2")
areas = np.linspace(100, 1000, 91)
plt.figure()
plt.plot(areas, [c.good_package_cost([a], 0.1, 0.30, 3, 50, 0.99) for a in areas], label="monolithic")
plt.plot(areas, [c.good_package_cost([a / 4 * 1.1] * 4, 0.1, 0.30, 3, 110, 0.99) for a in areas], label="4 chiplets")
plt.xlabel("total logic area (mm2)"); plt.ylabel("cost per good package ($)"); plt.legend(); plt.grid(True)
plt.show()

break-even area 562 mm2
